In [ ]:
import os
import json
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr
import requests


In [ ]:
load_dotenv(override=True)

openai_api_key = os.getenv('OPENAI_API_KEY')
if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set")

ollama_response = requests.get("http://localhost:11434").content

ollama = OpenAI(base_url="http://localhost:11434/v1", api_key="ollama")

openai = OpenAI()

In [ ]:
MODELS = {
    "OpenAI (GPT-6 Luna)": {
        "client": openai,
        "model": "gpt-6-luna",
        "supports_tools": True,
        "extra_params": {"reasoning_effort": "none"},
    },
    "Ollama (Llama 3.2 - Local)": {
        "client": ollama,
        "model": "llama3.2",
        "supports_tools": False,
        "extra_params": {},
    },
}

In [ ]:
system_prompt = """
You are a pragmatic, security-first Senior System Design Architect.
We deploy strictly on our own Linux VPS / bare-metal servers (using Docker, Nginx/Caddy, Redis, Postgres, Celery/BullMQ, etc.) — do NOT default to hyperscale cloud services like AWS Lambda, ECS, or GCP unless comparing them as an alternative.

When I ask how to design or scale a system, give me a practical, right-sized architecture for that exact scale with BOTH solutions below:

1. **Capacity & Hardware Sizing (VPS Specs)**:
   - What X concurrent requests actually means in RPS, CPU cores, RAM, GPU/VRAM, and network bandwidth.
   - Exact VPS specs needed (e.g., 4 vCPU, 16GB RAM, or GPU VPS requirements).

2. **Solution A: 100% Open-Source / Self-Hosted on VPS (Zero External Data Sharing)**:
   - Full architecture running inside our VPS (e.g., FastAPI/Node + Redis Queue + Worker pool + Open-Source models like PaddleOCR/Tesseract/vLLM/Ollama).
   - How concurrency is handled without crashing the VPS (async workers, job queues, rate limiting, connection pooling).

3. **Solution B: Hybrid VPS + Frontier API (or Cloud Comparison)**:
   - Running our backend/queue on our VPS while calling Frontier APIs (OpenAI, Anthropic, etc.) with async concurrency (`asyncio`/semaphores, retries, rate-limit handling).
   - Brief comparison against Managed Cloud only for reference.

4. **Security & Zero-PII Leakage**:
   - How to ensure PII (Personally Identifiable Information) never leaks to external Frontier APIs (e.g., local PII scrubbing/redaction on the VPS before external API calls, or routing PII documents strictly to Solution A's local open-source model).
   - VPS hardening (UFW firewall, reverse proxy TLS, secret management, encryption at rest).

5. **Breaking Point & Scaling Up**:
   - At what concurrency this single VPS will bottleneck, and how to scale next (vertical VPS upgrade vs. adding a second worker VPS behind Nginx load balancing).

## Conversation & Tool Guidelines
- Use the 5-section structure above for initial or full system design questions.
- For follow-up or targeted questions (e.g., tweaking one component, comparing trade-offs, or drafting an ADR), respond concisely and directly without repeating all 5 sections.
- Whenever the user asks to estimate, calculate, or compare LLM API costs or monthly spend, ALWAYS call the `estimate_llm_cost` tool (when available) instead of doing pricing math in your head. If the user doesn't specify token counts, state reasonable assumptions and pass them to the tool.
"""


In [ ]:
PRICING_PER_1M = {
    "gpt-6-luna": {"input": 0.15, "output": 0.60},
    "llama3.2-local": {"input": 0.00, "output": 0.00},
}

def estimate_llm_cost(model, requests_per_day, avg_input_tokens, avg_output_tokens):
    print(
        f"TOOL CALLED: estimate_llm_cost(model={model}, req/day={requests_per_day}, "
        f"in={avg_input_tokens}, out={avg_output_tokens})",
        flush=True,
    )
    key = model.lower()
    if key not in PRICING_PER_1M:
        supported = ", ".join(PRICING_PER_1M.keys())
        return f"Unknown model '{model}'. Supported models: {supported}"
    rates = PRICING_PER_1M[key]
    monthly_requests = requests_per_day * 30
    input_cost = (monthly_requests * avg_input_tokens / 1_000_000) * rates["input"]
    output_cost = (monthly_requests * avg_output_tokens / 1_000_000) * rates["output"]
    total_cost = input_cost + output_cost
    note = " (self-hosted on VPS; API cost is $0, budget for VPS/GPU hosting instead)" if total_cost == 0 else ""
    return (
        f"Estimated 30-day API cost for {key} "
        f"({monthly_requests:,} req/month, {avg_input_tokens:,} in / {avg_output_tokens:,} out tokens):\n"
        f"- Input cost: ${input_cost:.2f}\n"
        f"- Output cost: ${output_cost:.2f}\n"
        f"- Total monthly API cost: ${total_cost:.2f}{note}"
    )
        


In [ ]:
cost_function = {
    "name": "estimate_llm_cost",
    "description": "Calculate deterministic 30-day LLM API costs for a given model, daily request volume, and average input/output tokens.",
    "parameters": {
        "type": "object",
        "properties": {
            "model": {
                "type": "string",
                "enum": list(PRICING_PER_1M.keys()),
                "description": "The model to estimate API cost for.",
            },
            "requests_per_day": {
                "type": "number",
                "description": "Expected number of LLM requests per day.",
            },
            "avg_input_tokens": {
                "type": "integer",
                "description": "Average prompt/input tokens per request.",
            },
            "avg_output_tokens": {
                "type": "integer",
                "description": "Average completion/output tokens per request.",
            },
        },
        "required": ["model", "requests_per_day", "avg_input_tokens", "avg_output_tokens"],
        "additionalProperties": False,
    },
}

tools = [{"type": "function", "function": cost_function}]

In [ ]:
def handle_tool_calls(message):
    responses = []
    for tool_call in message.tool_calls:
        if tool_call.function.name == "estimate_llm_cost":
            args = json.loads(tool_call.function.arguments)
            result = estimate_llm_cost(
                model=args["model"],
                requests_per_day=args["requests_per_day"],
                avg_input_tokens=args["avg_input_tokens"],
                avg_output_tokens=args["avg_output_tokens"],
            )
            responses.append({
                "role": "tool",
                "content": result,
                "tool_call_id": tool_call.id,
            })
    return responses

In [ ]:
def chat(message, history, selected_model):
    history = [{"role": h["role"], "content": h["content"]} for h in history]
    messages = [{"role": "system", "content": system_prompt}] + history + [{"role": "user", "content": message}]

    config = MODELS[selected_model]
    client = config["client"]
    model = config["model"]

    while True:
        kwargs = {"model": model, "messages": messages, **config["extra_params"]}
        if config["supports_tools"]:
            kwargs["tools"] = tools

        with client.chat.completions.stream(**kwargs) as stream:
            for event in stream:
                if event.type == "content.delta":
                    yield event.snapshot
            response = stream.get_final_completion()

        if response.choices[0].finish_reason == "tool_calls":
            assistant_msg = response.choices[0].message
            tool_responses = handle_tool_calls(assistant_msg)
            messages.append({
                "role": "assistant",
                "content": assistant_msg.content,
                "tool_calls": [
                    {
                        "id": tc.id,
                        "type": "function",
                        "function": {"name": tc.function.name, "arguments": tc.function.arguments},
                    }
                    for tc in assistant_msg.tool_calls
                ],
            })
            messages.extend(tool_responses)
        else:
            break

In [ ]:
model_dropdown = gr.Dropdown(
    choices=list(MODELS.keys()),
    value=list(MODELS.keys())[0],
    label="Model",
)

gr.ChatInterface(
    fn=chat,
    type="messages",
    title="AI System & Cost Architect",
    description="Pragmatic, security-first VPS & AI architecture advisor with streaming, model switching, and deterministic cost estimation.",
    additional_inputs=[model_dropdown],
    additional_inputs_accordion=gr.Accordion(label="Model Settings", open=True),
    examples=[
        ["How do I handle 25 concurrent OCR requests on a VPS vs frontier API?", "OpenAI (GPT-6 Luna)"],
        ["Estimate our 30-day API cost for 5,000 requests/day using gpt-6-luna vs llama3.2-local.", "OpenAI (GPT-6 Luna)"],
        ["How should we scrub PII before calling an external LLM API?", "Ollama (Llama 3.2 - Local)"],
    ],
).launch()